In [52]:
import numpy as np
print(np.__version__)
import pandas as pd

2.3.4


# Transforms and Modelling Notebook

This notebook is for feature engineering, transformations, and machine learning modelling.


In [53]:
# Load the original dataset
df = pd.read_csv('../data/online_shoppers_intention.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


Dataset shape: (12330, 18)

Columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

First few rows:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## Applying closure to make sure the data is projected into the right space (simplex)

In [54]:
count_cols = ["Administrative", "Informational", "ProductRelated"]
duration_cols = ["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]

def closure(df, cols):
    subset = df[cols]
    sums = subset.sum(axis=1)

    closed = subset.div(sums.replace(0, pd.NA), axis=0)
    return closed

# Apply closure
df_counts_closed = closure(df, count_cols).add_suffix("_closed")
df_durs_closed   = closure(df, duration_cols).add_suffix("_closed")

# Append back to main dataframe
df_closure = pd.concat([df, df_counts_closed, df_durs_closed], axis=1)

df_closure.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,TrafficType,VisitorType,Weekend,Revenue,Administrative_closed,Informational_closed,ProductRelated_closed,Administrative_Duration_closed,Informational_Duration_closed,ProductRelated_Duration_closed
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,1,Returning_Visitor,False,False,0.0,0.0,1.0,<NA>,<NA>,<NA>
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,...,2,Returning_Visitor,False,False,0.0,0.0,1.0,0.0,0.0,1.0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,3,Returning_Visitor,False,False,0.0,0.0,1.0,<NA>,<NA>,<NA>
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,...,4,Returning_Visitor,False,False,0.0,0.0,1.0,0.0,0.0,1.0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,...,4,Returning_Visitor,True,False,0.0,0.0,1.0,0.0,0.0,1.0


In [55]:
df_closure.isna().sum()

Administrative                      0
Administrative_Duration             0
Informational                       0
Informational_Duration              0
ProductRelated                      0
ProductRelated_Duration             0
BounceRates                         0
ExitRates                           0
PageValues                          0
SpecialDay                          0
Month                               0
OperatingSystems                    0
Browser                             0
Region                              0
TrafficType                         0
VisitorType                         0
Weekend                             0
Revenue                             0
Administrative_closed               6
Informational_closed                6
ProductRelated_closed               6
Administrative_Duration_closed    720
Informational_Duration_closed     720
ProductRelated_Duration_closed    720
dtype: int64

In [56]:
# take the rows which have NaN in their *_closed columns and put them into a seprate df
df_closure_na = df_closure[df_closure.isna().any(axis=1)]
df_closure_na.head()

# drop the rows with NaN in their *_closed columns
df_closure = df_closure.dropna(subset=df_closure.columns[df_closure.columns.str.endswith("_closed")])

df_closure.shape

# Validation: Check closure is maintained (should sum to 1)
# Validation: Check closure is maintained (should sum to 1)
print("=== CLOSURE VALIDATION ===")
# Define closed column names properly
count_closed_cols = ["Administrative_closed", "Informational_closed", "ProductRelated_closed"]
dur_closed_cols = ["Administrative_Duration_closed", "Informational_Duration_closed", "ProductRelated_Duration_closed"]

print("Count closure check (should be ~1.0):")
print(df_closure[count_closed_cols].sum(axis=1).describe())

print("\nDuration closure check (should be ~1.0):")
print(df_closure[dur_closed_cols].sum(axis=1).describe())

=== CLOSURE VALIDATION ===
Count closure check (should be ~1.0):
count     11610.0
unique        2.0
top           1.0
freq      11606.0
dtype: float64

Duration closure check (should be ~1.0):
count     11610.0
unique        4.0
top           1.0
freq      11031.0
dtype: float64


### Handling 0's in data using Multiplicative replacement in order to maintain ratios 

In [57]:

def multiplicative_replace_closed(row, cols, delta=1e-6):
    """
    Multiplicative replacement for zeros in a closed compositional vector.
    row: pandas Series with the closed parts
    cols: list of column names (length D)
    delta: small positive constant
    """
    x = row[cols].to_numpy(dtype=float)
    D = len(x)
    zeros = (x == 0)
    z = zeros.sum()
    if z == 0:
        # ensure exact closure
        x = x / x.sum()
        return x
    
    # replace zeros with delta
    x[zeros] = delta
    
    # subtract mass proportionally from non-zeros
    mass_added = delta * z
    nonzero_idx = ~zeros
    nz_mass = x[nonzero_idx].sum()
    
    # if something breaks (all zeros), return uniform split
    if nz_mass <= 0:
        return np.ones(D) / D
    
    x[nonzero_idx] = x[nonzero_idx] * (1 - mass_added / nz_mass)
    
    # final closure
    x = x / x.sum()
    return x


### ILR 

In [58]:
def ilr_3parts(a, i, p):
    # assumes strictly positive values (after replacement)
    z1 = np.sqrt(2/3) * (np.log(p) - 0.5 * (np.log(a) + np.log(i)))
    z2 = np.sqrt(1/2) * (np.log(a) - np.log(i))
    return z1, z2


In [59]:
count_closed = ["Administrative_closed", "Informational_closed", "ProductRelated_closed"]
dur_closed   = ["Administrative_Duration_closed", "Informational_Duration_closed", "ProductRelated_Duration_closed"]

# Multiplicative replacement (counts group)
df_closure[count_closed] = df_closure.apply(
    lambda r: multiplicative_replace_closed(r, count_closed, delta=1e-6),
    axis=1, result_type="expand"
)

# Multiplicative replacement (durations group)
df_closure[dur_closed] = df_closure.apply(
    lambda r: multiplicative_replace_closed(r, dur_closed, delta=1e-6),
    axis=1, result_type="expand"
)

# ILR features for counts
df_closure[["ilr_counts_z1", "ilr_counts_z2"]] = df_closure.apply(
    lambda r: ilr_3parts(r[count_closed[0]], r[count_closed[1]], r[count_closed[2]]),
    axis=1, result_type="expand"
)

# ILR features for durations
df_closure[["ilr_durs_z1", "ilr_durs_z2"]] = df_closure.apply(
    lambda r: ilr_3parts(r[dur_closed[0]], r[dur_closed[1]], r[dur_closed[2]]),
    axis=1, result_type="expand"
)

df_closure.head()


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Administrative_closed,Informational_closed,ProductRelated_closed,Administrative_Duration_closed,Informational_Duration_closed,ProductRelated_Duration_closed,ilr_counts_z1,ilr_counts_z2,ilr_durs_z1,ilr_durs_z2
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.0,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.0,11.280316,0.0
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.0,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.0,11.280316,0.0
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.0,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.0,11.280316,0.0
5,0,0.0,0,0.0,19,154.216667,0.015789,0.024561,0.0,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.0,11.280316,0.0
8,0,0.0,0,0.0,2,37.000000,0.000000,0.100000,0.0,0.8,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.0,11.280316,0.0


In [60]:
# Validation: Check multiplicative replacement worked correctly
print("=== MULTIPLICATIVE REPLACEMENT VALIDATION ===")
print("Zeros in counts after replacement (should be 0):")
print((df_closure[count_closed] == 0).sum().sum())
print("\nZeros in durations after replacement (should be 0):")
print((df_closure[dur_closed] == 0).sum().sum())

print("\nClosure maintained after replacement:")
print("Count closure check (should be ~1.0):")
print(df_closure[count_closed].sum(axis=1).describe())
print("\nDuration closure check (should be ~1.0):")
print(df_closure[dur_closed].sum(axis=1).describe())


=== MULTIPLICATIVE REPLACEMENT VALIDATION ===
Zeros in counts after replacement (should be 0):
0

Zeros in durations after replacement (should be 0):
0

Closure maintained after replacement:
Count closure check (should be ~1.0):
count    1.161000e+04
mean     1.000000e+00
std      4.121666e-18
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

Duration closure check (should be ~1.0):
count    1.161000e+04
mean     1.000000e+00
std      8.047811e-18
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


In [61]:
# Validation: Check ILR transformation properties
print("=== ILR TRANSFORMATION VALIDATION ===")

# Verify ILR features are orthogonal (correlation should be ~0)
ilr_counts = df_closure[["ilr_counts_z1", "ilr_counts_z2"]]
ilr_durs = df_closure[["ilr_durs_z1", "ilr_durs_z2"]]

print("ILR counts correlation (should be ~0):")
print(ilr_counts.corr().iloc[0,1])
print("\nILR durations correlation (should be ~0):")
print(ilr_durs.corr().iloc[0,1])

print("\nILR features summary:")
print("Count ILR features:")
print(ilr_counts.describe())
print("\nDuration ILR features:")
print(ilr_durs.describe())

# Check for any infinite or NaN values
print("\nInfinite/NaN check:")
print("Count ILR - Infinite values:", np.isinf(ilr_counts).sum().sum())
print("Count ILR - NaN values:", ilr_counts.isna().sum().sum())
print("Duration ILR - Infinite values:", np.isinf(ilr_durs).sum().sum())
print("Duration ILR - NaN values:", ilr_durs.isna().sum().sum())


=== ILR TRANSFORMATION VALIDATION ===
ILR counts correlation (should be ~0):
-0.2671646893523972

ILR durations correlation (should be ~0):
-0.2782076632180781

ILR features summary:
Count ILR features:
       ilr_counts_z1  ilr_counts_z2
count   11610.000000   11610.000000
mean        7.531555       2.895482
std         3.463065       4.445052
min        -5.640158      -9.769040
25%         5.876868       0.000000
50%         6.798075       0.286707
75%        11.280316       7.955347
max        11.280316       9.769040

Duration ILR features:
        ilr_durs_z1   ilr_durs_z2
count  11610.000000  11610.000000
mean       7.696078      2.845467
std        3.451717      4.395344
min      -10.611029     -9.769040
25%        5.853577      0.000000
50%        7.085944      0.000000
75%       11.280316      7.716699
max       11.280316      9.769040

Infinite/NaN check:
Count ILR - Infinite values: 0
Count ILR - NaN values: 0
Duration ILR - Infinite values: 0
Duration ILR - NaN values: 0


In [62]:
df_closure

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Administrative_closed,Informational_closed,ProductRelated_closed,Administrative_Duration_closed,Informational_Duration_closed,ProductRelated_Duration_closed,ilr_counts_z1,ilr_counts_z2,ilr_durs_z1,ilr_durs_z2
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.000000,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.000000,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.000000,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
5,0,0.0,0,0.0,19,154.216667,0.015789,0.024561,0.000000,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
8,0,0.0,0,0.0,2,37.000000,0.000000,0.100000,0.000000,0.8,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,0.007143,0.029031,12.241717,0.0,...,0.053571,0.000001,0.946428,0.075177,0.000001,0.924822,6.790038,7.699523,6.632859,7.939108
12326,0,0.0,0,0.0,5,465.750000,0.000000,0.021333,0.000000,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
12327,0,0.0,0,0.0,6,184.250000,0.083333,0.086667,0.000000,0.0,...,0.000001,0.000001,0.999998,0.000001,0.000001,0.999998,11.280316,0.000000,11.280316,0.000000
12328,4,75.0,0,0.0,15,346.000000,0.000000,0.021053,0.000000,0.0,...,0.210526,0.000001,0.789473,0.178147,0.000001,0.821852,6.083257,8.667266,6.184254,8.549179


In [63]:


# One-hot encode VisitorType 
df_closure = pd.get_dummies(df_closure, columns=['VisitorType'])

# One-hot encode Month
df_closure = pd.get_dummies(df_closure, columns=['Month'])

# One-hot encode Browser
df_closure = pd.get_dummies(df_closure, columns=['Browser'])

# One-hot encode Region
df_closure = pd.get_dummies(df_closure, columns=['Region'])

# One-hot encode TrafficType
df_closure = pd.get_dummies(df_closure, columns=['TrafficType'])

# One-hot encode OS
df_closure = pd.get_dummies(df_closure, columns=['OperatingSystems'])



df_closure



,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,TrafficType_19,TrafficType_20,OperatingSystems_1,OperatingSystems_2,OperatingSystems_3,OperatingSystems_4,OperatingSystems_5,OperatingSystems_6,OperatingSystems_7,OperatingSystems_8
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.000000,0.0,...,False,False,False,True,False,False,False,False,False,False
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
5,0,0.0,0,0.0,19,154.216667,0.015789,0.024561,0.000000,0.0,...,False,False,False,True,False,False,False,False,False,False
8,0,0.0,0,0.0,2,37.000000,0.000000,0.100000,0.000000,0.8,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,0.007143,0.029031,12.241717,0.0,...,False,False,False,False,False,True,False,False,False,False
12326,0,0.0,0,0.0,5,465.750000,0.000000,0.021333,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
12327,0,0.0,0,0.0,6,184.250000,0.083333,0.086667,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
12328,4,75.0,0,0.0,15,346.000000,0.000000,0.021053,0.000000,0.0,...,False,False,False,True,False,False,False,False,False,False


In [ ]:
#for all columns in df_closure, if the datatype is bool, convert to int
bool_cols = df_closure.select_dtypes(include=["bool"]).columns
df_closure[bool_cols] = df_closure[bool_cols].astype(int)


In [65]:


def drop_highest_freq_reference(df: pd.DataFrame, groups: list):
    """
    For each categorical group already OHE'd, drop the dummy column
    corresponding to the highest-frequency category (reference).
    
    Args:
        df (pd.DataFrame): Input dataframe with OHE columns
        groups (list): List of base names for categorical groups
    
    Returns:
        pd.DataFrame: Dataframe with reference columns dropped
        dict: Mapping of group -> dropped reference category
    """
    dropped_refs = {}

    for group in groups:
       
        group_cols = [c for c in df.columns if c.startswith(group + "_")]
        if not group_cols:
            continue

        
        ref_col = df[group_cols].sum().idxmax()
        dropped_refs[group] = ref_col

       
        df = df.drop(columns=[ref_col])

    return df, dropped_refs

# Example usage:
groups = ["Browser", "OperatingSystems", "Region", "VisitorType", "Month", "TrafficType"]
df_new, refs = drop_highest_freq_reference(df_closure, groups)
print("Reference categories dropped:", refs)


Reference categories dropped: {'Browser': 'Browser_2', 'OperatingSystems': 'OperatingSystems_2', 'Region': 'Region_1', 'VisitorType': 'VisitorType_Returning_Visitor', 'Month': 'Month_May', 'TrafficType': 'TrafficType_2'}


In [66]:
target = "Revenue"
cont_feats = ["ilr_counts_z1","ilr_counts_z2","ilr_durs_z1","ilr_durs_z2",
              "PageValues","BounceRates","ExitRates"]
bin_feats  = ["Weekend","SpecialDay"]
ohe_feats  = [c for c in df_new.columns if any(c.startswith(g+"_") for g in groups)]
X = df_new[cont_feats + bin_feats + ohe_feats].copy()
y = df_new[target].astype(int).values

In [67]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_trs, X_vas = X_tr.copy(), X_va.copy()
X_trs[cont_feats] = scaler.fit_transform(X_tr[cont_feats])
X_vas[cont_feats] = scaler.transform(X_va[cont_feats])


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, classification_report
clf = LogisticRegression(penalty="l2", C=1.0, solver="liblinear", max_iter=2000)
clf.fit(X_trs, y_tr)


p_tr = clf.predict_proba(X_trs)[:,1]
p_va = clf.predict_proba(X_vas)[:,1]
print("AUC (train, val):", roc_auc_score(y_tr,p_tr), roc_auc_score(y_va,p_va))
print("LogLoss (train, val):", log_loss(y_tr,p_tr), log_loss(y_va,p_va))
print(classification_report(y_va, (p_va>=0.5).astype(int)))


AUC (train, val): 0.894612864272955 0.8863142493587064
LogLoss (train, val): 0.30407319484467993 0.30781400092197475
              precision    recall  f1-score   support

           0       0.89      0.98      0.93      1941
           1       0.75      0.38      0.50       381

    accuracy                           0.88      2322
   macro avg       0.82      0.68      0.72      2322
weighted avg       0.87      0.88      0.86      2322



In [68]:
# Save data for HMC_NUTS notebook
import pickle
import os

os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_trs.npy', X_trs.values)
np.save('../data/processed/X_vas.npy', X_vas.values)
np.save('../data/processed/y_tr.npy', y_tr)
np.save('../data/processed/y_va.npy', y_va)

with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../data/processed/cont_feats.pkl', 'wb') as f:
    pickle.dump(cont_feats, f)

with open('../data/processed/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X_trs.columns), f)

print("Data saved for HMC_NUTS notebook")
print(f"X_trs shape: {X_trs.shape}, y_tr shape: {y_tr.shape}")


Data saved for HMC_NUTS notebook
X_trs shape: (9288, 66), y_tr shape: (9288,)
